In [1]:
import os
import sys
import numpy as np
import time
import argparse
import multiprocessing as mp

In [2]:
def compute_point_dists(A, B):
    """ Compute pair-wise point distances in two matrices.

        Input:
            A: [np.ndarray, (N,3), np.float32]
                point cloud A
            B: [np.ndarray, (M,3), np.float32]
                point cloud B

        Output:
            dists: [np.ndarray, (N,M), np.float32]
                distance matrix
    """
    A = A[:, np.newaxis, :]
    B = B[np.newaxis, :, :]
    dists = np.linalg.norm(A-B, axis=-1)
    return dists

In [46]:
def worker(obj_name, point_ind, params, tolerance):
    scores, dists = params
    tmp_tolerance = np.zeros([V, A, D], dtype=np.float32)
    tic = time.time()
    for r in radius_list:
        dist_mask = (dists[point_ind] <= r)
        scores_in_ball = scores[dist_mask]
        pos_ratio = ((scores_in_ball > 0) & (scores_in_ball <= cfgs.mu_thresh)).mean(axis=0)
        tolerance_mask = (pos_ratio >= cfgs.pos_ratio_thresh)
        if tolerance_mask.sum() == 0:
            break
        tmp_tolerance[tolerance_mask] = r
    tolerance[point_ind] = tmp_tolerance
    toc = time.time()
    print("{}: point {} time".format(obj_name, point_ind), toc - tic)


In [3]:
root = 'F:\graspnet'
camera = 'kinect'

In [4]:
V = 300
A = 12
D = 4
radius_list = [0.001 * x for x in range(51)]
obj_list = ['%03d' % x for x in range(88)]

In [25]:
obj_name = '000'
pool_size=8
label_path = '{}_labels.npz'.format(obj_name)
label = np.load(os.path.join(root, 'grasp_label', label_path))
points = label['points']
scores = label['scores']

tolerance = mp.Manager().dict()
dists = compute_point_dists(points, points)
params = (scores, dists)

pool = []
process_cnt = 0
work_list = [x for x in range(len(points))]
# for _ in range(pool_size):
#     point_ind = work_list.pop(0)
#     pool.append(mp.Process(target=worker, args=(obj_name, point_ind, params, tolerance)))
# [p.start() for p in pool]

In [26]:
tmp_tolerance = np.zeros([V, A, D], dtype=np.float32)
point_ind = work_list.pop(0)
print(point_ind)
dist_mask = (dists[point_ind] <= 0.01)
print(dist_mask)
scores_in_ball = scores[dist_mask]

0
[ True False False ... False False False]


In [27]:
scores_in_ball

array([[[[-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ],
         ...,
         [-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ]],

        [[-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ],
         ...,
         [-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ]],

        [[-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ],
         ...,
         [-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ]],

        ...,

        [[-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ],
         ...,
         [-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ],
         [-1. , -1. , -1. , -1. ]],

        [[-1. , -1. , -1. ,  0.6],
         [-1. , -1. , -1. , -1. ],
         [-1

In [48]:
np.unique(scores_in_ball)

array([-1. ,  0.1,  0.2,  0.3,  0.4,  0.5,  0.6,  0.7,  0.8,  0.9,  1. ],
      dtype=float32)

In [41]:
pos_ratio = ((scores_in_ball > 0) & (scores_in_ball <= 0.4)).mean(0)

In [42]:
pos_ratio

array([[[0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        ...,
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ]],

       [[0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        ...,
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ]],

       [[0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        ...,
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.      

In [44]:
tolerance_mask = (pos_ratio >= 0.8)
tolerance_mask

array([[[False, False, False, False],
        [False, False, False, False],
        [False, False, False, False],
        ...,
        [False, False, False, False],
        [False, False, False, False],
        [False, False, False, False]],

       [[False, False, False, False],
        [False, False, False, False],
        [False, False, False, False],
        ...,
        [False, False, False, False],
        [False, False, False, False],
        [False, False, False, False]],

       [[False, False, False, False],
        [False, False, False, False],
        [False, False, False, False],
        ...,
        [False, False, False, False],
        [False, False, False, False],
        [False, False, False, False]],

       ...,

       [[False, False, False, False],
        [False, False, False, False],
        [False, False, False, False],
        ...,
        [False, False, False, False],
        [False, False, False, False],
        [False, False, False, False]],

       [[False,

In [45]:
if tolerance_mask.sum() == 0:
    print(0)

In [ ]:
tmp_tolerance[tolerance_mask] = 

array([[0.        , 0.09321812, 0.03977671, ..., 0.07529936, 0.24645472,
        0.03749225],
       [0.09321812, 0.        , 0.08347244, ..., 0.08535087, 0.18791506,
        0.05760296],
       [0.03977671, 0.08347244, 0.        , ..., 0.06320298, 0.2511437 ,
        0.03233859],
       ...,
       [0.07529936, 0.08535087, 0.06320298, ..., 0.        , 0.20631582,
        0.06164267],
       [0.24645472, 0.18791506, 0.2511437 , ..., 0.20631582, 0.        ,
        0.22506313],
       [0.03749225, 0.05760296, 0.03233859, ..., 0.06164267, 0.22506313,
        0.        ]], shape=(3459, 3459), dtype=float32)

In [7]:
np.unique(scores)

array([-1. ,  0.1,  0.2,  0.3,  0.4,  0.5,  0.6,  0.7,  0.8,  0.9,  1. ],
      dtype=float32)

In [47]:
scores.shape

(3459, 300, 12, 4)

In [10]:
tolerance

<DictProxy object, typeid 'dict' at 0x2101a5d6f20>

In [50]:
tol = np.load(r"F:\graspnet\tolerance\000_tolerance.npy")

In [52]:
np.unique(tol)

array([0.   , 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008,
       0.009, 0.01 , 0.011, 0.012, 0.013, 0.014, 0.015, 0.016, 0.017,
       0.018, 0.019, 0.02 , 0.021, 0.022, 0.023, 0.024, 0.025, 0.026,
       0.027, 0.028, 0.029, 0.03 , 0.031, 0.032, 0.033, 0.034, 0.035,
       0.036, 0.037, 0.038, 0.039, 0.04 , 0.041, 0.042, 0.043, 0.044,
       0.045, 0.046, 0.047, 0.048, 0.049, 0.05 ], dtype=float32)

In [53]:
tol.shape

(3459, 300, 12, 4)

In [39]:
arr = np.array([
    [80, 100, 60],
    [90, 70, 50]
])
print(arr.mean(0))

[85. 85. 55.]


In [40]:
print(arr.mean(1))

[80. 70.]
